# Bodo DataFrames

[Bodo DataFrames](https://www.bodo.ai/bodo-dataframes) is a high performance DataFrame library for large scale Python data processing and AI/ML use cases. It functions as a drop-in replacement for Pandas; simply replace:

``` py 
import pandas as pd
```

with:

``` py
import bodo.pandas as pd
```

to automatically scale and accelerate Pandas workloads. Bodo also provides additional Pandas-compatible APIs for simplifying and scaling AI workloads, a just-in-time (JIT) compiler for custom transformations, as well as an integrated SQL engine for extra flexibility.

Under the hood, Bodo DataFrames uses lazy evaluation to optimize sequences of Pandas operations, streams data through operators to enable processing larger-than-memory datasets, and leverages MPI-based high-performance computing technology for efficient parallel execution that can easily scale from laptop to large cluster.


## Included in This Notebook

This notebook demonstrates some of the core functionality of Bodo DataFrames for data engineering, data science, AI and ML applications. 
It has three main sections: 
1. Simple AI inference workflows
2. Scalable dataset management with Iceberg
3. Accelerating Pandas with a one-line-change. 


## 0. Environment Setup

The dataset used in sections 2 and 3 is from the [New York City Taxi and Limousine Commission](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) and contains trips from taxi and rideshare apps from 2019-2023. 
This notebook uses a subset of the 2019 data (\~1 GiB, Parquet format). In addition to the Taxi dataset, 
section 3 also uses a small dataset of [weather observations from Central Park](https://github.com/toddwschneider/nyc-taxi-data/blob/c65ad8332a44f49770644b11576c0529b40bbc76/data/central_park_weather.csv) (\~0.5 MiB CSV file). 
The data is hosted in a public S3 bucket and downloading the data first eliminates variability due to network speeds.

In [1]:
import os

os.environ["BODO_NUM_WORKERS"] = "8"
os.environ["BODO_PLATFORM_CACHE_LOCATION"] = "~/.bodo_cache"

In [2]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

bucket_name = "bodo-example-data"


def download_data_s3(path_to_s3: str, local_data_dir: str = "data") -> str:
    """Download the dataset from S3. If already exists, skip download."""
    file_name = os.path.basename(path_to_s3)
    local_path = os.path.join(local_data_dir, file_name)

    if os.path.exists(local_path):
        return local_path

    print("Downloading dataset from S3...")

    s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

    if not os.path.exists(local_data_dir):
        os.mkdir(local_data_dir)

    s3.download_file(bucket_name, path_to_s3, local_path)
    return local_path


# Download the weather data (CSV)
download_data_s3(
    "nyc-taxi/central_park_weather.csv",
)

# Download Taxi data (parquet files)
pq_files = [
    "nyc-taxi/fhvhv_tripdata/fhvhv_tripdata_2019-02.parquet",
    "nyc-taxi/fhvhv_tripdata/fhvhv_tripdata_2019-03.parquet",
]

for file in pq_files:
    download_data_s3(file, local_data_dir="data/fhvhv_tripdata")

In [3]:
# Filter warnings on workers
# These are performance warnings due to the small number of data files.
# Skip this cell to see the raw output.
import os
import warnings

import bodo.pandas as pd
import bodo.spawn.spawner as spawner

spawner.submit_func_to_workers(lambda: warnings.filterwarnings("ignore"), [])

# Filter Frontend warnings
warnings.filterwarnings("ignore")

## 1. Simple AI inference workflows

Bodo DataFrames provides an extended version of the Pandas API for simplifying and scaling common AI workflows.
In this example, we show how you can create and query vector embeddings using Bodo DataFrames and S3 Vectors.
Bodo DataFrames parallelizes calls to S3 Vectors APIs when putting vectors, 
making it ideal for loading large chunks of data at a time. The cells below uses a preconfigured vector bucket. To set up your own vector bucket/index refer to the first 2 steps of [this tutorial](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors-getting-started.html).

For more details about specific AI toolkit functions, refer our API docs pages:
* [BodoSeries.ai.embed](https://docs.bodo.ai/latest/api_docs/dataframe_lib/series/ai/embed/)
* [BodoDataFrame.to_s3_vectors](https://docs.bodo.ai/latest/api_docs/dataframe_lib/dataframe/to_s3_vectors/)
* [BodoSeries.ai.query_s3_vectors](https://docs.bodo.ai/latest/api_docs/dataframe_lib/series/ai/query_s3_vectors/)

In [4]:
import getpass

openai_api_key = getpass.getpass("OPENAI_API_KEY")

OPENAI_API_KEY ········


In [5]:
from pathlib import Path
from dotenv import load_dotenv

dotenv_path = Path.home() / Path("shared/analyst/bodo/.env")
assert load_dotenv(dotenv_path)

In [6]:
# Setup (create vector index)

import random

embedding_model = "text-embedding-3-small"
embedding_dim = 1536

vector_bucket_name = "bodo-pytorch-conf-demo-bucket"
region = "us-east-2"
index_name = f"movie-summaries{random.randint(100000, 999999)}"

s3vectors = boto3.client("s3vectors", region_name=region)

_ = s3vectors.create_index(
    vectorBucketName=vector_bucket_name,
    indexName=index_name,
    dataType="float32",
    dimension=embedding_dim,
    distanceMetric="cosine",
)

In [7]:
# Create embeddings using OpenAI API

texts = [
    "Star Wars: A farm boy joins rebels to fight an evil empire in space",
    "Jurassic Park: Scientists create dinosaurs in a theme park that goes wrong",
    "Finding Nemo: A father fish searches the ocean to find his lost son",
]
keys = ["Star Wars", "Jurassic Park", "Finding Nemo"]
genres = ["scifi", "scifi", "family"]


df = pd.DataFrame({"key": keys, "text": texts, "genre": genres})
df["data"] = df.text.ai.embed(model=embedding_model, api_key=openai_api_key)

df["data"]

0    [-0.05204673  0.0732903  -0.00437872 ...  0.00...
1    [-0.00484369  0.06850401  0.00451336 ... -0.00...
2    [ 0.01458554  0.04918431 -0.01516628 ...  0.00...
Name: data, dtype: list<item: double>[pyarrow]

In [8]:
# Write embeddings into vector index with metadata.

df["metadata"] = df.apply(
    lambda row: {"source_text": row.text, "genre": row.genre}, axis=1
)
df.to_s3_vectors(
    vector_bucket_name=vector_bucket_name, index_name=index_name, region=region
)

In [9]:
# Query the vector index (with filtering)

input_text = "adventures in space"
df = pd.DataFrame({"text": [input_text]})
df["data"] = df.text.ai.embed(model="text-embedding-3-small", api_key=openai_api_key)
out = df.data.ai.query_s3_vectors(
    vector_bucket_name=vector_bucket_name,
    index_name=index_name,
    region=region,
    topk=3,
    filter={"genre": "scifi"},
    return_distance=True,
    return_metadata=True,
)

display(out)

,keys,distances,metadata
0,['Star Wars' 'Jurassic Park'],[0.62190527 0.73654616],"[""{'source_text': 'Star Wars: A farm boy joins..."


In [10]:
# Cleanup (delete index)

_ = s3vectors.delete_index(vectorBucketName=vector_bucket_name, indexName=index_name)

## 2. Scalable dataset management with Iceberg

Bodo DataFrames provides simple and scalable APIs for reading and writing data to Iceberg tables, an open source table format which provides an extra layer of scalable dataset management on top of raw files. 
One benefit of using Iceberg is the Time Travel feature, which let's you inspect the state of a table at a previous point in time, so you can track your data as it changes.

For more details about Bodo's Iceberg support, refer to our [Iceberg documentation](https://docs.bodo.ai/latest/quick_start/quickstart_local_iceberg/).

In [11]:
import bodo.pandas as pd
import pyiceberg
from bodo.io.iceberg.catalog.dir import DirCatalog

warehouse_loc = "./iceberg_warehouse"
table_name = "fhvhv_tripdata"

In [12]:
# Load a portion of the NYC Taxi Data into Iceberg

df = pd.read_parquet("data/fhvhv_tripdata/fhvhv_tripdata_2019-02.parquet")
df = df[
    [
        "hvfhs_license_num",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "dropoff_datetime",
        "pickup_datetime",
    ]
].head(5)

df.to_iceberg(table_name, location=warehouse_loc)

out_df = pd.read_iceberg(table_name, location=warehouse_loc)
out_df

,hvfhs_license_num,PULocationID,DOLocationID,trip_miles,dropoff_datetime,pickup_datetime
0,HV0003,245,251,2.45,2019-02-01 00:14:57,2019-02-01 00:05:18
1,HV0003,216,197,1.71,2019-02-01 00:49:39,2019-02-01 00:41:29
2,HV0005,261,234,5.01,2019-02-01 01:28:29,2019-02-01 00:51:34
3,HV0005,87,87,0.34,2019-02-01 00:07:16,2019-02-01 00:03:51
4,HV0005,87,198,6.84,2019-02-01 00:39:56,2019-02-01 00:09:44


In [13]:
# Add more rows to the table

df = pd.read_parquet("data/fhvhv_tripdata/fhvhv_tripdata_2019-03.parquet")
df = df[
    [
        "hvfhs_license_num",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "dropoff_datetime",
        "pickup_datetime",
    ]
].head(5)

df.to_iceberg(table_name, location=warehouse_loc, append=True)

out_df = pd.read_iceberg(table_name, location=warehouse_loc)
out_df

,hvfhs_license_num,PULocationID,DOLocationID,trip_miles,dropoff_datetime,pickup_datetime
0,HV0003,245,251,2.45,2019-02-01 00:14:57,2019-02-01 00:05:18
1,HV0003,216,197,1.71,2019-02-01 00:49:39,2019-02-01 00:41:29
2,HV0005,261,234,5.01,2019-02-01 01:28:29,2019-02-01 00:51:34
3,HV0005,87,87,0.34,2019-02-01 00:07:16,2019-02-01 00:03:51
4,HV0005,87,198,6.84,2019-02-01 00:39:56,2019-02-01 00:09:44
5,HV0004,36,80,2.18,2019-03-01 00:28:51,2019-03-01 00:13:55
6,HV0004,37,232,3.66,2019-03-01 00:43:03,2019-03-01 00:23:58
7,HV0005,25,62,2.53,2019-03-01 00:15:09,2019-03-01 00:03:37
8,HV0003,65,262,9.34,2019-03-01 00:50:43,2019-03-01 00:29:46
9,HV0003,140,196,7.88,2019-03-01 01:20:47,2019-03-01 00:58:56


In [14]:
# Use PyIceberg to get a history of the dataset over time.

catalog = DirCatalog(
    None,
    **{
        pyiceberg.catalog.WAREHOUSE_LOCATION: warehouse_loc,
    },
)

table = catalog.load_table(f".{table_name}")
history = table.history()
prev_snapshot = history[0].snapshot_id

history

[SnapshotLogEntry(snapshot_id=4850196048922924379, timestamp_ms=1761093989485),
 SnapshotLogEntry(snapshot_id=1183952406153453001, timestamp_ms=1761093992604)]

In [15]:
# Use Time Travel to query both versions of the table

df = pd.read_iceberg(
    table_name, location=warehouse_loc, snapshot_id=prev_snapshot
)
display(df)

current_snapshot = history[1].snapshot_id

df = pd.read_iceberg(
    table_name, location=warehouse_loc, snapshot_id=current_snapshot
)
display(df)

,hvfhs_license_num,PULocationID,DOLocationID,trip_miles,dropoff_datetime,pickup_datetime
0,HV0003,245,251,2.45,2019-02-01 00:14:57,2019-02-01 00:05:18
1,HV0003,216,197,1.71,2019-02-01 00:49:39,2019-02-01 00:41:29
2,HV0005,261,234,5.01,2019-02-01 01:28:29,2019-02-01 00:51:34
3,HV0005,87,87,0.34,2019-02-01 00:07:16,2019-02-01 00:03:51
4,HV0005,87,198,6.84,2019-02-01 00:39:56,2019-02-01 00:09:44


,hvfhs_license_num,PULocationID,DOLocationID,trip_miles,dropoff_datetime,pickup_datetime
0,HV0003,245,251,2.45,2019-02-01 00:14:57,2019-02-01 00:05:18
1,HV0003,216,197,1.71,2019-02-01 00:49:39,2019-02-01 00:41:29
2,HV0005,261,234,5.01,2019-02-01 01:28:29,2019-02-01 00:51:34
3,HV0005,87,87,0.34,2019-02-01 00:07:16,2019-02-01 00:03:51
4,HV0005,87,198,6.84,2019-02-01 00:39:56,2019-02-01 00:09:44
5,HV0004,36,80,2.18,2019-03-01 00:28:51,2019-03-01 00:13:55
6,HV0004,37,232,3.66,2019-03-01 00:43:03,2019-03-01 00:23:58
7,HV0005,25,62,2.53,2019-03-01 00:15:09,2019-03-01 00:03:37
8,HV0003,65,262,9.34,2019-03-01 00:50:43,2019-03-01 00:29:46
9,HV0003,140,196,7.88,2019-03-01 01:20:47,2019-03-01 00:58:56


In [16]:
# Cleanup (delete local warehouse)

import shutil

shutil.rmtree(warehouse_loc)

## 3. Accelerating Pandas code with a one-line-change

Bodo DataFrames automatically parallelizes and optimizes workloads written in Pandas.
This example uses a representative data engineering workload for creating trip summaries using Central Park weather observations (CSV) and NYC Taxi/Rideshare data (Parquet),
highlighting Bodo DataFrames performance on key transformations such as read parquet, datetime manipulation, merge, groupby and sort. 
We first measure the performance of Bodo DataFrames, then run in Pandas to see an improvement. 
This gap becomes larger as we scale to more data and add more cores.

For more details about supported DataFrames features, refer to our [API documentation](https://docs.bodo.ai/latest/api_docs/dataframe_lib/).

In [17]:
from time import perf_counter

weather_dataset = "data/central_park_weather.csv"
taxi_dataset = [
    "data/fhvhv_tripdata/fhvhv_tripdata_2019-02.parquet", 
    "data/fhvhv_tripdata/fhvhv_tripdata_2019-03.parquet"
]

def get_monthly_travels_weather(weather_dataset : str, taxi_dataset : str, out_file : str, pd):
    """ Run the full workload and write results to Parquet. """
    start = perf_counter()
    central_park_weather_observations = pd.read_csv(
        weather_dataset,
        parse_dates=["DATE"],
    )
    central_park_weather_observations = central_park_weather_observations.rename(
        columns={"DATE": "date", "PRCP": "precipitation"}, copy=False
    )
    fhvhv_tripdata = pd.read_parquet(taxi_dataset)

    central_park_weather_observations["date"] = central_park_weather_observations[
        "date"
    ].dt.date
    fhvhv_tripdata["date"] = fhvhv_tripdata["pickup_datetime"].dt.date
    fhvhv_tripdata["month"] = fhvhv_tripdata["pickup_datetime"].dt.month
    fhvhv_tripdata["hour"] = fhvhv_tripdata["pickup_datetime"].dt.hour
    fhvhv_tripdata["weekday"] = fhvhv_tripdata["pickup_datetime"].dt.dayofweek.isin(
        [0, 1, 2, 3, 4]
    )

    monthly_trips_weather = fhvhv_tripdata.merge(
        central_park_weather_observations, on="date", how="inner"
    )
    monthly_trips_weather["date_with_precipitation"] = (
        monthly_trips_weather["precipitation"] > 0.1
    )

    def get_time_bucket(t):
        bucket = "other"
        if t in (8, 9, 10):
            bucket = "morning"
        elif t in (11, 12, 13, 14, 15):
            bucket = "midday"
        elif t in (16, 17, 18):
            bucket = "afternoon"
        elif t in (19, 20, 21):
            bucket = "evening"
        return bucket

    monthly_trips_weather["time_bucket"] = monthly_trips_weather.hour.map(
        get_time_bucket
    )
    monthly_trips_weather = monthly_trips_weather.groupby(
        [
            "PULocationID",
            "DOLocationID",
            "month",
            "weekday",
            "date_with_precipitation",
            "time_bucket",
        ],
        as_index=False,
    ).agg({"hvfhs_license_num": "count", "trip_miles": "mean"})
    monthly_trips_weather = monthly_trips_weather.sort_values(
        by=[
            "PULocationID",
            "DOLocationID",
            "month",
            "weekday",
            "date_with_precipitation",
            "time_bucket",
        ]
    )
    monthly_trips_weather = monthly_trips_weather.rename(
        columns={
            "hvfhs_license_num": "trips",
            "trip_miles": "avg_distance",
        },
        copy=False,
    )

    monthly_trips_weather.to_parquet(out_file)
    end = perf_counter()
    print("Total E2E time:", (end - start))

### JIT Warmup

The function above uses `Series.map` to compute a user defined function. By default, APIs for applying custom functions like `Series.map` or  `DataFrame.apply` use JIT compilation to avoid Python overheads. Since Bodo's JIT modules are lazily loaded the first time one of these APIs is called, we run a simple "warmup" cell below to make sure JIT load is not factored in to the workload time.

In [18]:
import bodo.pandas

S = bodo.pandas.Series(range(10))
_ = S.map(lambda x: x + 1)

In [19]:
get_monthly_travels_weather(
    weather_dataset, taxi_dataset, out_file="bodo_monthly_trips_weather_pandas.pq", pd=bodo.pandas
)

Total E2E time: 9.677385398001206


### Compilation Caching

JIT compilation adds overheads that can be noticeable on small data sizes. By default JIT results are cached, so subsequent runs of the workload will be even faster.

In [20]:
get_monthly_travels_weather(
    weather_dataset, taxi_dataset, out_file="bodo_monthly_trips_weather_pandas.pq", pd=bodo.pandas
)

Total E2E time: 6.4788018349991034


### Comparing to Pandas

Run the cell below to see the comparison to Pandas. 
Without any code changed, Bodo accelerates the workflow by ~8x using 8 workers.

In [21]:
import pandas

get_monthly_travels_weather(
    weather_dataset, taxi_dataset, out_file="pandas_monthly_trips_weather.pq", pd=pandas
)

Total E2E time: 52.0725275650002
